RetailPulse 360

Notebook 12 — Store Performance Segmentation

Goal: Segment all 190 stores by real performance behavior — sales velocity, trend, category
mix, volatility — to answer "which stores need attention, which are thriving, which are
declining." Pivoted from generic "customer segmentation" since RetailPulse 360 has no
individual customer-level data anywhere (a real, honest gap — see notes) — this uses real
data we actually have rather than fabricating a synthetic customer layer.

Input: sales.csv, stores.csv, skus.csv, store_personalities.csv, inventory_turnover_summary.csv
Output: store_segments.csv

In [10]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully.")

Libraries imported successfully.


In [11]:
# 2. LOAD INPUTS
# ============================================================

BASE_PATH = "/kaggle/input/datasets/hamaz911/notebook-12-datasets/"

sales = pd.read_csv(BASE_PATH + "sales.csv", parse_dates=["date"])
stores = pd.read_csv(BASE_PATH + "stores.csv")
skus = pd.read_csv(BASE_PATH + "skus.csv")
store_personalities = pd.read_csv(BASE_PATH + "store_personalities.csv")
inventory = pd.read_csv(BASE_PATH + "inventory_turnover_summary.csv")

print("Loaded:")
for name, df in [("sales", sales), ("stores", stores), ("skus", skus),
                  ("store_personalities", store_personalities), ("inventory", inventory)]:
    print(f"  {name}: {df.shape}")

Loaded:
  sales: (2459464, 4)
  stores: (191, 18)
  skus: (3296, 13)
  store_personalities: (1115, 12)
  inventory: (13678, 7)


In [12]:
# 3. DATA QUALITY — VALIDATE INPUTS
# ============================================================

print("Missing values:")
for name, df in [("stores", stores), ("inventory", inventory)]:
    m = df.isna().sum()
    if m.sum() > 0:
        print(f"  {name}: {dict(m[m > 0])}")
print("(only known gaps should show: STY-ECOM personality, 9 Fully Dead rows)")

print("\nReferential integrity:")
print("  sales -> stores orphans:", (~sales["store_id"].isin(stores["store_id"])).sum())
print("  sales -> skus orphans:", (~sales["sku_id"].isin(skus["sku_id"])).sum())

Missing values:
  stores: {'store_size': np.int64(1), 'rossmann_store_id': np.int64(1), 'dow_mon': np.int64(1), 'dow_tue': np.int64(1), 'dow_wed': np.int64(1), 'dow_thu': np.int64(1), 'dow_fri': np.int64(1), 'dow_sat': np.int64(1), 'dow_sun': np.int64(1), 'promo_lift': np.int64(1), 'trend_pct_per_year': np.int64(1), 'volatility_cv': np.int64(1), 'holiday_lift': np.int64(1)}
  inventory: {'active_daily_velocity': np.int64(9), 'days_of_supply': np.int64(9)}
(only known gaps should show: STY-ECOM personality, 9 Fully Dead rows)

Referential integrity:
  sales -> stores orphans: 0
  sales -> skus orphans: 0


In [13]:
# 4. BUILD STORE PERFORMANCE FEATURES
# ============================================================
# Real, comparable metrics per store: revenue, growth trend (from
# Rossmann personality), volatility, and category concentration
# (how specialized vs. balanced a store's sales mix is).

sales_j = sales.merge(skus[["sku_id", "category", "price_pkr"]], on="sku_id", how="left")
sales_j["revenue"] = sales_j["units_sold"] * sales_j["price_pkr"]

store_features = sales_j.groupby("store_id").agg(
    total_revenue=("revenue", "sum"),
    total_units=("units_sold", "sum"),
).reset_index()

# Category concentration: share of revenue from a store's single
# best-selling category. High = specialized (relies heavily on one
# category), Low = balanced across categories.
cat_share = sales_j.groupby(["store_id", "category"])["revenue"].sum().reset_index()
top_cat_share = cat_share.groupby("store_id")["revenue"].apply(lambda x: x.max() / x.sum()).reset_index()
top_cat_share.columns = ["store_id", "top_category_concentration"]

store_features = store_features.merge(top_cat_share, on="store_id", how="left")
store_features = store_features.merge(
    stores[["store_id", "store_size", "region", "trend_pct_per_year", "volatility_cv"]],
    on="store_id", how="left"
)

# Inventory health context (from Notebook 06)
inv_health = inventory.groupby("store_id").agg(
    avg_days_of_supply=("days_of_supply", lambda x: x[np.isfinite(x)].mean())
).reset_index()
store_features = store_features.merge(inv_health, on="store_id", how="left")

print("Store features built:", store_features.shape)
print(store_features.describe())
print("\nMissing values:", store_features.isna().sum()[store_features.isna().sum() > 0].to_dict())

Store features built: (190, 9)
       total_revenue   total_units  top_category_concentration  \
count   1.900000e+02    190.000000                  190.000000   
mean    7.807832e+07  13173.084211                    0.569611   
std     4.743967e+07   7990.969928                    0.025160   
min     2.283934e+07   3874.000000                    0.483219   
25%     2.951495e+07   4992.000000                    0.555015   
50%     9.845268e+07  16839.500000                    0.572498   
75%     1.165625e+08  19715.250000                    0.586067   
max     1.705150e+08  28563.000000                    0.658038   

       trend_pct_per_year  volatility_cv  avg_days_of_supply  
count          190.000000     190.000000          190.000000  
mean             0.037997       0.263178           17.946944  
std              0.062763       0.056584            5.440805  
min             -0.242100       0.123400            7.437940  
25%              0.013750       0.224825           13.14032

In [14]:
# 5. STANDARDIZE FEATURES, THEN FIND THE RIGHT NUMBER OF CLUSTERS
# ============================================================

feature_cols = ["total_revenue", "total_units", "top_category_concentration",
                 "trend_pct_per_year", "volatility_cv", "avg_days_of_supply"]

X = store_features[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Standardized feature check (mean should be ~0, std should be ~1):")
print(pd.DataFrame(X_scaled, columns=feature_cols).describe().loc[["mean", "std"]].round(2))

# Elbow method: try K from 2 to 8, record within-cluster sum of squares
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

print("\nInertia (within-cluster spread) by K:")
for k, inertia in zip(K_range, inertias):
    print(f"  K={k}: {inertia:.1f}")

Standardized feature check (mean should be ~0, std should be ~1):
      total_revenue  total_units  top_category_concentration  \
mean            0.0         -0.0                         0.0   
std             1.0          1.0                         1.0   

      trend_pct_per_year  volatility_cv  avg_days_of_supply  
mean                 0.0            0.0                -0.0  
std                  1.0            1.0                 1.0  

Inertia (within-cluster spread) by K:
  K=2: 650.0
  K=3: 544.8
  K=4: 456.7
  K=5: 402.5
  K=6: 359.5
  K=7: 328.1
  K=8: 297.4


In [15]:
# 6. FIT FINAL MODEL (K=4) AND EXAMINE EACH SEGMENT
# ============================================================

FINAL_K = 4
km_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
store_features["segment"] = km_final.fit_predict(X_scaled)

print("Stores per segment:")
print(store_features["segment"].value_counts().sort_index())

print("\nSegment profile (mean values per feature, per segment):")
segment_profile = store_features.groupby("segment")[feature_cols].mean().round(2)
print(segment_profile)

Stores per segment:
segment
0     13
1     40
2    101
3     36
Name: count, dtype: int64

Segment profile (mean values per feature, per segment):
         total_revenue  total_units  top_category_concentration  \
segment                                                           
0         2.984835e+07      5048.62                        0.57   
1         3.047456e+07      5040.42                        0.59   
2         1.207777e+08     20364.32                        0.57   
3         2.859233e+07      4967.81                        0.54   

         trend_pct_per_year  volatility_cv  avg_days_of_supply  
segment                                                         
0                     -0.07           0.38               13.66  
1                      0.05           0.25               13.27  
2                      0.04           0.26               22.26  
3                      0.05           0.24               12.60  


In [16]:
# 6b. FIX — NORMALIZE REVENUE WITHIN SIZE TIER (find surprises, not size)
# ============================================================
# Raw revenue/units mostly just re-derive store_size, which we already
# know. Normalizing performance RELATIVE TO each store's own size-tier
# average lets clustering find genuinely new signal: over/under-performers
# within their own peer group, not just "big stores make more money."

store_features["revenue_vs_size_peer_avg"] = store_features.groupby("store_size")["total_revenue"].transform(
    lambda x: x / x.mean()
)

# Rebuild the feature set: drop raw revenue/units (proxies for size),
# keep the relative performance measure + the genuinely distinct signals
feature_cols_v2 = ["revenue_vs_size_peer_avg", "top_category_concentration",
                    "trend_pct_per_year", "volatility_cv", "avg_days_of_supply"]

X2 = store_features[feature_cols_v2].values
X2_scaled = StandardScaler().fit_transform(X2)

inertias_v2 = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X2_scaled)
    inertias_v2.append(km.inertia_)

print("Inertia by K (corrected features):")
for k, inertia in zip(range(2,9), inertias_v2):
    print(f"  K={k}: {inertia:.1f}")

Inertia by K (corrected features):
  K=2: 742.0
  K=3: 617.5
  K=4: 509.6
  K=5: 441.3
  K=6: 387.7
  K=7: 363.9
  K=8: 341.7


In [19]:
# 7. FIT FINAL MODEL (K=4, CORRECTED FEATURES) AND EXAMINE SEGMENTS
# ============================================================

FINAL_K = 4
km_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
store_features["segment"] = km_final.fit_predict(X2_scaled)

print("Stores per segment:")
print(store_features["segment"].value_counts().sort_index())

print("\nSegment profile (corrected features):")
print(store_features.groupby("segment")[feature_cols_v2].mean().round(2))

print("\nStore size breakdown per segment (checking this ISN'T just re-deriving size again):")
print(pd.crosstab(store_features["segment"], store_features["store_size"]))

Stores per segment:
segment
0    69
1    60
2    38
3    23
Name: count, dtype: int64

Segment profile (corrected features):
         revenue_vs_size_peer_avg  top_category_concentration  \
segment                                                         
0                            0.98                        0.57   
1                            1.08                        0.59   
2                            0.97                        0.54   
3                            0.90                        0.57   

         trend_pct_per_year  volatility_cv  avg_days_of_supply  
segment                                                         
0                      0.02           0.26               22.93  
1                      0.09           0.25               15.72  
2                      0.04           0.23               13.26  
3                     -0.04           0.36               16.54  

Store size breakdown per segment (checking this ISN'T just re-deriving size again):
store_siz

In [20]:
# 8. LABEL SEGMENTS AND SAVE
# ============================================================

SEGMENT_LABELS = {
    0: "Steady Performers",
    1: "Rising Stars",
    2: "Lean & Consistent",
    3: "At-Risk / Declining",
}
store_features["segment_label"] = store_features["segment"].map(SEGMENT_LABELS)

store_segments = store_features.merge(stores[["store_id", "city"]], on="store_id", how="left")

print("Final segment counts:")
print(store_segments["segment_label"].value_counts())

store_segments.to_csv("store_segments.csv", index=False)
print("\nSaved store_segments.csv —", store_segments.shape)

Final segment counts:
segment_label
Steady Performers      69
Rising Stars           60
Lean & Consistent      38
At-Risk / Declining    23
Name: count, dtype: int64

Saved store_segments.csv — (190, 13)
